In [1]:
# Get raw data
with open('input/23.txt', 'r') as f:
    rawinput = f.read().strip()

In [2]:
from intcode import Program  # From Day 9

In [3]:
# Part 1
class Computer(Program):
    instr = [int(i) for i in rawinput.split(',')]
    def __init__(self, cid, verbose=False):
        self.cid = cid
        super().__init__(self.instr, [cid], verbose)
        
    def do_exec(self):
        pass
    
    def do_step(self):
        opmode = self.instr[self.ptr]
        opcode = opmode % 100
        self.ops[opcode](*self.process_params(opmode))
        
    def do_input(self, addr, val):
        if len(self.input_val) == 0:
            self.input_val += [-1]
        if self.verbose:  print(f"Ptr={self.ptr:2d}:  Set address {addr[0]} from INPUT to {self.input_val[0]}")
        self.maybe_extend_memory(addr[0])
        self.instr[addr[0]] = self.input_val.pop(0)
        self.ptr += 2
    
    def add_input(self, inputs):
        if self.verbose:  print(f"Adding inputs {inputs} to computer {self.cid}:")
        self.input_val += inputs
    
    def collect_output(self):
        if self.verbose and len(self.output)//3:
            print(f"Collecting outputs ({len(self.output)//3} packets) from computer {self.cid}:")
        return [[self.output.pop(0) for j in range(3)]
                for i in range(len(self.output)//3)]
    
class Network(object):
    def __init__(self, n, verbose=False):
        self.n = n
        self.verbose = verbose
        self.computers = [Computer(i, verbose) for i in range(n)]
        
    def do_exec(self, reset=True):
        if reset:  
            for i in self.computers:
                i.reset()
        result = None
        while result is None:
            for i in self.computers:
                if self.verbose:
                    print(f"Computer {i.cid}:")
                i.do_step()
            for k,l,m in [j for i in self.computers for j in i.collect_output()]:
                if k==255:
                    result = m
                    break
                else:
                    self.computers[k].add_input([l,m])
                    
        return result
    
network = Network(50)
network.do_exec()

17849

In [4]:
# Part 2
class Network(object):
    def __init__(self, n, verbose=False):
        self.n = n
        self.verbose = verbose
        self.computers = [Computer(i, verbose) for i in range(n)]
        self.nat_memory = None
        self.nat_last_send = None
        self.nat_idle_flags = [5 for i in range(n)]
        
    def do_exec(self, reset=True):
        if reset:  
            for i in self.computers:
                i.reset()
        while True:
            for i in self.computers:
                if self.verbose:
                    print(f"Computer {i.cid}:")
                if i.ptr==73 and len(i.input_val)==0:
                    self.nat_idle_flags[i.cid] -= 1
                i.do_step()
            if (outputs:=[j for i in self.computers for j in i.collect_output()]):
                self.nat_idle_flags = [5 for i in range(self.n)]
                for k,l,m in outputs:
                    if k==255:
                        self.nat_memory = [l,m]
                    else:
                        self.computers[k].add_input([l,m])
            if all([i<=0 for i in self.nat_idle_flags]):
                if self.nat_last_send is not None and self.nat_memory[1]==self.nat_last_send[1]:
                    result = self.nat_memory[1]
                    break
                else:
                    self.computers[0].add_input(self.nat_memory)
                    self.nat_last_send = self.nat_memory
                    self.nat_idle_flags = [5 for i in range(self.n)]
                    
        return result
    
network = Network(50)
network.do_exec()

12235